# PM₂.₅ Compliance Report Against WHO Guidelines

**Goal:** Produce a monthly air quality summary suitable for a Local Authority
Annual Status Report, checking PM₂.₅ against WHO 2021 guidelines and UK limit values.

**API keys required:** None (AURN and AQE are freely accessible)

**Aeolus features demonstrated:**
- `find_sites()` for nearest-monitor discovery
- `download()` for data retrieval
- `metrics.aq_stats()` for regulatory statistics
- `metrics.aqi_summary()` with UK DAQI index
- `metrics.aqi_check_who()` for WHO guideline compliance
- `viz.plot_calendar()` for calendar heatmap
- `viz.plot_timeseries()` with guideline overlay

In [ ]:
import aeolus
from aeolus import metrics, viz
from datetime import datetime

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend for notebook execution
import matplotlib.pyplot as plt

## 1. Find the Nearest Monitor

A local authority officer needs data for their borough. We use `find_sites()`
to find the closest AURN or AQE monitor to a given location.

In [ ]:
# Find nearest monitors to a borough centre (e.g., Camden Town Hall)
# Search across free UK regulatory networks
sites = aeolus.find_sites(
    ["AURN", "AQE"],
    near=(51.5392, -0.1426),  # Camden
    radius_km=10,
)

print(f"Found {len(sites)} monitors within 10km")
sites[["site_code", "site_name", "source_network", "distance_km"]].head()

In [ ]:
# Select the nearest site (already sorted by distance)
target_site = sites.iloc[0]["site_code"]
target_network = sites.iloc[0]["source_network"]
target_name = sites.iloc[0]["site_name"]

print(f"Using: {target_name} ({target_site}) from {target_network}")
print(f"Distance: {sites.iloc[0]['distance_km']:.1f} km")

## 2. Download Data

In [ ]:
# Download a full year of data
data = aeolus.download(
    target_network,
    sites=[target_site],
    start_date=datetime(2024, 1, 1),
    end_date=datetime(2024, 12, 31),
)

# Filter to PM2.5 — try fallback sites if the nearest doesn't have it
pm25 = data[data["measurand"] == "PM2.5"]

if pm25.empty and len(sites) > 1:
    print(f"No PM2.5 data at {target_site}, trying other nearby sites...")
    for i in range(1, min(len(sites), 5)):
        alt_site = sites.iloc[i]["site_code"]
        alt_network = sites.iloc[i]["source_network"]
        alt_data = aeolus.download(
            alt_network, sites=[alt_site],
            start_date=datetime(2024, 1, 1), end_date=datetime(2024, 12, 31),
        )
        pm25 = alt_data[alt_data["measurand"] == "PM2.5"]
        if not pm25.empty:
            target_site, target_network = alt_site, alt_network
            target_name, data = sites.iloc[i]["site_name"], alt_data
            print(f"Using {target_name} ({target_site}) from {target_network}")
            break

print(f"PM\u2082.\u2085 records: {len(pm25):,}")
if not pm25.empty:
    print(f"Date range: {pm25['date_time'].min()} to {pm25['date_time'].max()}")

## 3. Data Capture Assessment

Data capture rate is critical for regulatory reporting. DEFRA requires ≥75%
for annual statistics to be considered valid.

In [ ]:
# aq_stats automatically calculates data capture
annual = metrics.aq_stats(pm25, pollutant="PM2.5")

if annual.empty:
    print("No PM2.5 statistics available (insufficient data).")
else:
    capture = annual["data_capture"].iloc[0]
    print(f"Annual data capture: {capture:.1%}")
    status = "\u2705 Valid" if capture >= 0.75 else "\u26a0\ufe0f Below threshold"
    print(f"Status: {status}")
    display(annual[["site_code", "year", "data_capture", "annual_mean",
                     "max_daily_mean", "p95"]])

## 4. WHO Guideline Compliance

The WHO 2021 guidelines set a PM₂.₅ annual guideline of 5 µg/m³, with
interim targets (IT-1 through IT-4) for countries making progress. The UK
limit value is currently 20 µg/m³ (annual mean).

In [ ]:
# Check all WHO targets
for target in ["AQG", "IT-4", "IT-3", "IT-2", "IT-1"]:
    result = metrics.aqi_check_who(pm25, target=target)
    if result.empty:
        continue
    pm_row = result[result["pollutant"] == "PM2.5"]
    if not pm_row.empty:
        row = pm_row.iloc[0]
        status = "\u2705" if row["meets_guideline"] else "\u274c"
        print(f"{status} {target:>4}: {row['guideline_value']:.0f} \u00b5g/m\u00b3 "
              f"(measured: {row['mean_concentration']:.1f} \u00b5g/m\u00b3)")

## 5. UK DAQI Distribution

Calculate daily AQI values and show the distribution across DAQI bands.
This tells the public how many "good" vs "poor" air quality days they experienced.

In [ ]:
# Daily AQI summary
daily_aqi = metrics.aqi_summary(pm25, index="UK_DAQI", freq="D")

# Count days in each category
pm_aqi = daily_aqi[daily_aqi["pollutant"] == "PM2.5"]
if not pm_aqi.empty:
    category_counts = pm_aqi["aqi_category"].value_counts().sort_index()
    print("Days in each DAQI band:")
    for category, count in category_counts.items():
        print(f"  {category}: {count} days")
else:
    print("No daily AQI values calculated.")

## 6. Time Series with Guideline

Plot the full year's data with a WHO guideline overlay to visually
identify periods of poor air quality.

In [ ]:
# Time series with WHO annual guideline
# Strip timezone for matplotlib compatibility
pm25_plot = pm25.copy()
pm25_plot["date_time"] = pm25_plot["date_time"].dt.tz_localize(None)

if not pm25_plot.empty:
    fig = viz.plot_timeseries(
        pm25_plot,
        pollutants=["PM2.5"],
        guideline=5.0,
        guideline_label="WHO AQG (5 \u00b5g/m\u00b3)",
        title=f"PM\u2082.\u2085 at {target_name} ({target_site}) \u2014 2024",
    )
    plt.show()
else:
    print("No PM2.5 data available for timeseries plot.")

## 7. Calendar Heatmap

A calendar heatmap provides an intuitive view of daily pollution levels
across the year. Missing data appears as blank cells.

In [ ]:
# Calendar heatmap of daily mean PM2.5
daily = metrics.time_average(pm25, freq="D")

# Strip timezone for matplotlib
daily_plot = daily.copy()
daily_plot["date_time"] = daily_plot["date_time"].dt.tz_localize(None)

fig = viz.plot_calendar(
    daily_plot,
    pollutant="PM2.5",
    year=2024,
    title=f"Daily PM\u2082.\u2085 \u2014 {target_name} (2024)",
)
plt.show()

## 8. Monthly Summary Table

Produce a table suitable for inclusion in a Local Authority Annual Status Report.

In [ ]:
# Monthly statistics
monthly = metrics.time_average(pm25, freq="ME")
monthly_pm = monthly[monthly["measurand"] == "PM2.5"].copy()
monthly_pm["month"] = monthly_pm["date_time"].dt.strftime("%B")

summary = monthly_pm[["month", "value", "data_capture"]].rename(
    columns={"value": "mean_pm25_ugm3", "data_capture": "data_capture_%"}
)
summary["data_capture_%"] = (summary["data_capture_%"] * 100).round(1)
summary["mean_pm25_ugm3"] = summary["mean_pm25_ugm3"].round(1)

print(f"Monthly PM\u2082.\u2085 Summary \u2014 {target_name} ({target_site})")
print("=" * 50)
display(summary)

## Summary

This notebook demonstrated a complete compliance reporting workflow:

1. **Site discovery** with nearest-monitor search
2. **Data capture assessment** using `aq_stats()`
3. **WHO guideline checking** across all interim targets
4. **DAQI band distribution** for public health messaging
5. **Calendar heatmap** for visual communication
6. **Monthly summary table** for regulatory reports

### For a full Annual Status Report
- Download 5 years for trend analysis (see `metrics.trend()`)
- Include NO₂ and PM₁₀ alongside PM₂.₅
- Compare multiple sites across the borough